# 01 - Data exploration
Coverage, gaps and distributions of `metrics_5m` (read-only).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
import config
from pipeline import data, clean, features, signals

raw = data.load_metrics_5m(config.db_path())
bars = clean.clean_metrics(raw)
feats = signals.add_signals(features.build_features(bars))
raw.shape, bars.shape

In [ ]:
# coverage per instrument
bars.groupby(['standardized_name', 'asset_class']).agg(first=('slot_end', 'min'), last=('slot_end', 'max'), bars=('slot_end', 'size'), with_data=('has_data', 'sum'))

In [ ]:
# data availability by UTC hour
bars.assign(hour=bars['slot_end'].dt.hour).pivot_table(index='hour', columns='standardized_name', values='has_data', aggfunc='mean').round(2)

In [ ]:
cols = ['log_return', 'rel_spread_mean', 'realised_vol_pct', 'flow_imbalance', 'book_imbalance', 'burst_count']
bars.groupby('standardized_name')[cols].describe().T

In [ ]:
# null share per column
bars[bars['has_data']].isna().mean().sort_values(ascending=False).head(20)

In [ ]:
sig = signals.signal_columns(feats)
feats[sig + features.target_columns(feats)].corr().round(3)